# SensorScheduler PPO Training (Step 5B)

Train the policy network that picks `(rgb_fps, thermal_fps)` for each scheduler tick.

**Inputs:** the repo (cloned from GitHub).
**Output:** `runs/sensor_scheduler/policy.zip` -- saved to Google Drive for retrieval.

Architecture: Phase B per `docs/ARCHITECTURE.md:189-280` (tiny MLP, ~3,500 params).

## 1. Install dependencies

In [ ]:
!pip install -q 'stable-baselines3[extra]' gymnasium ultralytics pyyaml websockets numpy nest_asyncio
import stable_baselines3, gymnasium
print('SB3:', stable_baselines3.__version__, '| Gymnasium:', gymnasium.__version__)

## 2. Clone the repo from GitHub

Adjust `BRANCH` if you want a different branch (e.g. `main`).

**If the repo is private**, set `GITHUB_TOKEN` to a Personal Access Token (GitHub Settings -> Developer settings -> Personal access tokens; scope `repo`). Leave it empty for a public repo.

In [ ]:
import os, shutil, subprocess

REPO_URL = 'https://github.com/aniket-27507/Sanjay_MK2.git'
BRANCH = 'main'
GITHUB_TOKEN = ''  # set only if the repo is private
WORKDIR = '/content/sanjay_mk2'

auth_url = (
    REPO_URL.replace('https://', f'https://x-access-token:{GITHUB_TOKEN}@')
    if GITHUB_TOKEN else REPO_URL
)

if os.path.exists(WORKDIR):
    shutil.rmtree(WORKDIR)
subprocess.run(['git', 'clone', '--branch', BRANCH, '--depth', '1', auth_url, WORKDIR], check=True)
os.chdir(WORKDIR)
print('cwd:', os.getcwd())
print('scenarios:', len(os.listdir('config/scenarios')))
print('Step 5A files present:',
      os.path.exists('src/single_drone/sensor_scheduler_env.py'),
      os.path.exists('scripts/train_sensor_scheduler.py'))

## 2b. (Optional) Mount Drive for checkpoint persistence

Even when cloning from GitHub, Drive is useful for saving `policy.zip` so the trained weights survive a Colab session reset and you can resume next time. Skip this cell to skip persistence.

In [ ]:
DRIVE_AVAILABLE = False
try:
    from google.colab import drive
    drive.mount('/content/drive')
    DRIVE_AVAILABLE = True
    print('Drive mounted.')
except Exception as e:
    print('Drive mount skipped:', e)

## 3. Resume-from-checkpoint setup

If a previous run saved `policy.zip` to Drive, copy it into `runs/sensor_scheduler/` so PPO can resume.

In [ ]:
DRIVE_CHECKPOINT = '/content/drive/MyDrive/SanjayMK2/runs/sensor_scheduler/policy.zip'
LOCAL_RUNS = 'runs/sensor_scheduler'
os.makedirs(LOCAL_RUNS, exist_ok=True)
if DRIVE_AVAILABLE and os.path.exists(DRIVE_CHECKPOINT):
    shutil.copy(DRIVE_CHECKPOINT, os.path.join(LOCAL_RUNS, 'policy.zip'))
    print('Resuming from Drive checkpoint.')
else:
    print('No Drive checkpoint found -- starting fresh.')

## 4. Train

200k steps is a reasonable first pass on a T4 (~30-60 min). Bump to 1M for a polished run.

In [ ]:
!python scripts/train_sensor_scheduler.py \
    --total-steps 200000 \
    --episode-duration 60 \
    --auto-resume \
    --tb

## 5. Push the trained policy back to Drive

Skipped automatically if Drive isn't mounted.

In [ ]:
if DRIVE_AVAILABLE:
    DRIVE_OUT_DIR = '/content/drive/MyDrive/SanjayMK2/runs/sensor_scheduler'
    os.makedirs(DRIVE_OUT_DIR, exist_ok=True)
    shutil.copy(os.path.join(LOCAL_RUNS, 'policy.zip'), os.path.join(DRIVE_OUT_DIR, 'policy.zip'))
    print('Pushed to', DRIVE_OUT_DIR)
else:
    print('Drive not mounted -- download policy.zip from the Files panel manually.')

## 6. Quick eval: trained policy vs. heuristic

Run a single fresh scenario with each policy and compare cumulative reward + compute cost.

In [ ]:
import nest_asyncio
nest_asyncio.apply()  # allow our env to run inside the Jupyter event loop

from stable_baselines3 import PPO
from src.single_drone.sensor_scheduler_env import SensorSchedulerEnv
from src.single_drone.sensor_scheduler_rl import encode_action

model = PPO.load(os.path.join(LOCAL_RUNS, 'policy.zip'))

def run_episode(use_trained: bool, seed: int = 0):
    env = SensorSchedulerEnv(episode_duration_sec=60.0, seed=seed)
    obs, _ = env.reset()
    total_r = 0.0; n_steps = 0; rgb_fires = 0; thermal_fires = 0
    while True:
        if use_trained:
            action, _ = model.predict(obs, deterministic=True)
        else:
            action = encode_action(15, 0)  # day-patrol heuristic default
        obs, r, terminated, truncated, info = env.step(int(action))
        total_r += r; n_steps += 1
        if info['rgb_fps'] > 0: rgb_fires += 1
        if info['thermal_fps'] > 0: thermal_fires += 1
        if terminated or truncated:
            break
    return dict(reward=total_r, steps=n_steps, rgb_active=rgb_fires, thermal_active=thermal_fires)

for seed in range(3):
    a = run_episode(use_trained=False, seed=seed)
    b = run_episode(use_trained=True, seed=seed)
    print(f'seed={seed}  heuristic={a}  trained={b}')